# **Flipkart Order Issue Resolution & Customer Support Analytics**



## **Your Engineering Handover**
You have joined the Order Support Operations team as a **Junior Python Developer**. The support lead spends hours switching between order records, finance refund logs, and customer complaints to understand which cases need attention. You have been asked to build a reproducible **order issue analysis workflow** in this notebook.

Your deliverable is **not an app or dashboard**. It is a trustworthy set of cleaned data tables, an order-level investigation report, and a concise evidence-based handover to Support Operations.

**How to work:** Make your own copy in Google Drive (`File → Save a copy in Drive`), upload the three provided files, complete the ten milestones in order, and run the assessment cell at the end. Write your own implementation in the code cells. Keep every required named output available for the next task and for assessment.

**Input files (only three):** `orders.csv` · `refunds.xlsx` · `customer_complaints.json`  
**Data:** Fictional training records; not actual Flipkart customer records.

> **Important:** The assessment at the end checks key output contracts and consistency. Passing it is not proof that every business calculation or written explanation is correct; reviewers may also inspect selected notebooks.

### **Important Note: Guidelines for Using AI Tools**



You are allowed to use AI tools such as ChatGPT to understand concepts, debug errors, and get assistance with **specific code snippets or individual problems**.

However, **using AI to generate the complete project solution in one go is not permitted**.

You are expected to develop the solution milestone by milestone, apply your understanding of Python and Pandas, and independently verify your code and outputs.

**Remember:** The goal of this project is not just to complete the tasks but to develop practical problem-solving, debugging, and data analysis skills that are expected in real-world development roles.

## **Workspace setup · Data intake**
Run the setup cell and upload the **three input files** when requested. Files stay in your current Colab session; if the runtime restarts, upload them again. Keep original filenames unchanged.

[Link to Dataset](https://drive.google.com/file/d/1F0o02Vzev9ar6OB3-TVWC7EGJcyiBKRv/view?usp=drive_link)

In [20]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path('/content') if Path('/content').is_dir() else Path.cwd()
REQUIRED_FILES = ['orders.csv', 'refunds.xlsx', 'customer_complaints.json']

missing = [name for name in REQUIRED_FILES if not (DATA_DIR / name).exists()]
if missing:
    try:
        from google.colab import files
        print('Please upload:', ', '.join(missing))
        files.upload()
    except ImportError:
        print(f'Place these files in {DATA_DIR}: {missing}')

from pathlib import Path
import zipfile
import pandas as pd

DATA_DIR = Path("/content")

REQUIRED_FILES = [
    "orders.csv",
    "refunds.xlsx",
    "customer_complaints.json"
]

# 1. Extract all valid ZIP files
def extract_zip_files():
    for zip_path in DATA_DIR.rglob("*.zip"):
        try:
            with zipfile.ZipFile(zip_path, "r") as z:
                if not z.namelist():
                    print("ZIP is empty:", zip_path.name)
                    continue

                z.extractall(DATA_DIR)
                print("Extracted:", zip_path.name)

        except (zipfile.BadZipFile, OSError):
            print("Invalid or damaged ZIP:", zip_path.name)


# 2. Find each required dataset, including in subfolders
def find_file(filename):
    matches = [
        p for p in DATA_DIR.rglob(filename)
        if p.is_file()
    ]
    return matches[0] if matches else None


# 3. Extract ZIP files already in Colab
extract_zip_files()

# 4. Check which datasets are available
file_paths = {
    name: find_file(name)
    for name in REQUIRED_FILES
}

missing = [
    name for name, path in file_paths.items()
    if path is None
]

# 5. Upload missing datasets
if missing:
    from google.colab import files

    print("\nMissing datasets:", ", ".join(missing))
    print("Upload the ACTUAL dataset files, not an empty ZIP.")
    print("You can select multiple files in the upload window.")

    uploaded = files.upload()

    # Extract any newly uploaded valid ZIP files
    extract_zip_files()

    # Search again after upload
    file_paths = {
        name: find_file(name)
        for name in REQUIRED_FILES
    }

# 6. Verify all required files
missing = [
    name for name, path in file_paths.items()
    if path is None
]

if missing:
    raise FileNotFoundError(
        f"Still missing: {missing}\n"
        "Please upload the actual orders.csv, refunds.xlsx, "
        "and customer_complaints.json files."
    )

# 7. Print the final file locations
DATA_DIR = file_paths["orders.csv"].parent

print("\nAll 3 dataset files are available!")
print("Dataset folder:", DATA_DIR)

for name, path in file_paths.items():
    print(f"{name}: {path}")

still_missing = [name for name in REQUIRED_FILES if not (DATA_DIR / name).exists()]
if still_missing:
    raise FileNotFoundError(f'Missing input files: {still_missing}')
print('All 3 source files are available.')

ZIP is empty: flipkart_project_datasets (3).zip
Extracted: flipkart_project_datasets (7).zip
ZIP is empty: flipkart_project_datasets (1).zip
ZIP is empty: flipkart_project_datasets.zip
ZIP is empty: flipkart_project_datasets (6).zip
ZIP is empty: flipkart_project_datasets (5).zip
ZIP is empty: flipkart_project_datasets (4).zip
Extracted: flipkart_project_datasets (8).zip
ZIP is empty: flipkart_project_datasets (2).zip

All 3 dataset files are available!
Dataset folder: /content
orders.csv: /content/orders.csv
refunds.xlsx: /content/refunds.xlsx
customer_complaints.json: /content/customer_complaints.json
All 3 source files are available.


## Milestone 1 · Establish the order baseline
**From the team:** The operations lead has asked for an initial quality and volume briefing before accepting the three files into the reporting workflow. You are responsible for understanding what is in the Orders source, not interpreting business issues yet.

**Your responsibility:** Read `orders.csv` and inspect its structure. Report record counts, distinct order and customer counts, the column types, and missing values. Show a sample that helps a reviewer understand the data.

**Handover evidence:** Keep the raw DataFrame under **`orders_df`**. Show a clearly labeled baseline summary and missing-value review.

In [22]:

# Milestone 1: Establish the Order Data Baseline

import pandas as pd
from IPython.display import display

# Load the orders dataset
orders_df = pd.read_csv(DATA_DIR / "orders.csv")

# Display sample records
print("First 5 Orders")
display(orders_df.head())

print("Last 5 Orders")
display(orders_df.tail())

# Check dataset structure
print("Dataset Shape:", orders_df.shape)
print("Total Records:", len(orders_df))
print("Total Columns:", len(orders_df.columns))
print("Column Names:", orders_df.columns.tolist())

print("\nColumn Data Types")
display(orders_df.dtypes.to_frame("Data Type"))

# Count unique orders and customers
print("\nUnique Orders:", orders_df["order_id"].nunique())
print("Unique Customers:", orders_df["customer_id"].nunique())

# Check missing values
print("\nMissing Value Summary")
display(orders_df.isnull().sum().to_frame("Missing Values"))

print("Total Missing Values:", orders_df.isnull().sum().sum())


First 5 Orders


,order_id,customer_id,customer_city,order_date,promised_delivery_date,actual_delivery_date,product_category,order_amount,order_status
0,FKO000702,CUS00237,Bengaluru,2026-05-26,2026-05-30,NaN,Grocery,36082,Cancelled
1,FKO000636,CUS00689,Hyderabad,2026-04-19,2026-04-25,2026-04-24,Grocery,47326,Delivered
2,FKO001570,CUS00222,Pune,2026-01-23,2026-01-30,NaN,Home & Kitchen,43254,Shipped
3,FKO001303,CUS00035,Lucknow,2026-02-12,2026-02-17,2026-02-18,Home & Kitchen,22215,Delivered
4,FKO001208,CUS00619,Lucknow,2026-07-08,2026-07-11,2026-07-09,Home & Kitchen,2512,Delivered


Last 5 Orders


,order_id,customer_id,customer_city,order_date,promised_delivery_date,actual_delivery_date,product_category,order_amount,order_status
1619,FKO000997,CUS00509,Bengaluru,2026-05-29,2026-06-01,NaN,Grocery,24350,Pending
1620,FKO001239,CUS00213,Jaipur,2026-03-30,2026-04-04,2026-04-09,Books,63113,Delivered
1621,FKO000507,CUS00746,Mumbai,2026-06-27,2026-06-29,2026-06-27,Fashion,60457,Delivered
1622,FKO000560,CUS00618,Indore,2026-02-08,2026-02-10,2026-02-11,Fashion,7686,DELIVERED
1623,FKO000282,CUS00202,Bengaluru,2026-07-10,2026-07-15,2026-07-16,Appliances,59343,Delivered


Dataset Shape: (1624, 9)
Total Records: 1624
Total Columns: 9
Column Names: ['order_id', 'customer_id', 'customer_city', 'order_date', 'promised_delivery_date', 'actual_delivery_date', 'product_category', 'order_amount', 'order_status']

Column Data Types


,Data Type
order_id,object
customer_id,object
customer_city,object
order_date,object
promised_delivery_date,object
actual_delivery_date,object
product_category,object
order_amount,int64
order_status,object



Unique Orders: 1600
Unique Customers: 673

Missing Value Summary


,Missing Values
order_id,0
customer_id,0
customer_city,0
order_date,0
promised_delivery_date,0
actual_delivery_date,492
product_category,0
order_amount,0
order_status,0


Total Missing Values: 492


## Milestone 2 · Onboard finance and customer support data
**From the team:** The finance team exports refunds to a spreadsheet, while customer support maintains a JSON complaint log. The team wants assurance that both files can be used alongside the orders data.

**Your responsibility:** Load and inspect both sources, assess their shapes and fields, and describe the current distribution of refund outcomes and complaint types. Identify any data quality concerns worth revisiting during cleaning.

**Handover evidence:** Keep raw DataFrames as **`refunds_df`** and **`complaints_df`**. Present counts and readable category/status summaries.

In [23]:

# Milestone 2: Understand Refund and Complaint Records

import pandas as pd
from IPython.display import display

# 1. Load the datasets
refunds_df = pd.read_excel(DATA_DIR / "refunds.xlsx")
complaints_df = pd.read_json(DATA_DIR / "customer_complaints.json")

# 2. Display sample records
print("First 5 Refund Records")
display(refunds_df.head())

print("First 5 Complaint Records")
display(complaints_df.head())

# 3. Inspect dataset structures
print("Refund Dataset Shape:", refunds_df.shape)
print("Refund Columns:", refunds_df.columns.tolist())
display(refunds_df.dtypes.to_frame("Data Type"))

print("Complaint Dataset Shape:", complaints_df.shape)
print("Complaint Columns:", complaints_df.columns.tolist())
display(complaints_df.dtypes.to_frame("Data Type"))

# 4. Analyze refund statuses
print("\nRefund Status Distribution")
display(refunds_df["refund_status"].value_counts(dropna=False).to_frame("Count"))

# 5. Analyze complaint types
print("\nComplaint Type Distribution")
display(complaints_df["complaint_type"].value_counts(dropna=False).to_frame("Count"))

# 6. Analyze complaint resolution statuses
print("\nComplaint Status Distribution")
display(complaints_df["complaint_status"].value_counts(dropna=False).to_frame("Count"))

# 7. Check missing values
print("\nMissing Values in Refunds")
display(refunds_df.isnull().sum().to_frame("Missing Values"))

print("\nMissing Values in Complaints")
display(complaints_df.isnull().sum().to_frame("Missing Values"))


First 5 Refund Records


,refund_id,order_id,refund_amount,refund_date,refund_status
0,RFD000291,FKO000086,15535,2026-06-16,Completed
1,RFD000103,FKO000927,4051,2026-07-01,Completed
2,RFD000518,FKO000907,8761,2026-02-24,Completed
3,RFD000217,FKO000233,16436,2026-04-12,Failed
4,RFD000800,FKO000474,2047,2026-05-02,Completed


First 5 Complaint Records


,complaint_id,order_id,customer_id,complaint_date,complaint_type,complaint_status
0,CMP001122,FKO001270,CUS00287,2026-02-08,Product Quality,Resolved
1,CMP000032,FKO001072,CUS00559,2026-03-20,Delivery Delay,resolved
2,CMP000582,FKO000243,CUS00684,2026-08-28,Cancellation Issue,Open
3,CMP000622,FKO001525,CUS00646,2026-08-28,Delivery Delay,Open
4,CMP000846,FKO000828,CUS00149,2026-03-22,Product Quality,In Progress


Refund Dataset Shape: (1371, 5)
Refund Columns: ['refund_id', 'order_id', 'refund_amount', 'refund_date', 'refund_status']


...

,Count
complaint_status,
Resolved,584
Open,488
In Progress,357
resolved,43
in progress,27
open,26



Missing Values in Refunds


,Missing Values
refund_id,0
order_id,0
refund_amount,0
refund_date,0
refund_status,0



Missing Values in Complaints


,Missing Values
complaint_id,0
order_id,0
customer_id,0
complaint_date,0
complaint_type,0
complaint_status,0


## Milestone 3 · Prepare reliable order records
**From the team:** Operations has noticed repeated IDs, inconsistent order labels, and delivery-date issues. Without cleaning, the team may miscount orders or report misleading delays.

**Your responsibility:** Produce a clean order-level source. Keep one record per order ID, normalize order statuses, ensure order amounts are numeric, and make order and delivery dates usable for comparisons. Treat non-delivered orders with no actual delivery date differently from invalid/missing dates on delivered orders. Explain your handling of ambiguous values.

**Handover evidence:** Create **`orders_clean`**, preserving key source fields: `order_id`, `customer_id`, `customer_city`, `order_date`, `promised_delivery_date`, `actual_delivery_date`, `product_category`, `order_amount`, `order_status`. Include a concise cleaning note.

In [24]:

# Milestone 3: Clean and Standardize Orders

# 1. Create a copy of the original data
orders_clean = orders_df.copy()

# 2. Check duplicate order IDs
print("Duplicate order IDs before cleaning:",
      orders_clean["order_id"].duplicated().sum())

# Keep one record per order ID
orders_clean = orders_clean.drop_duplicates(
    subset="order_id", keep="first"
).copy()

# 3. Clean text and standardize order status
text_columns = [
    "order_id", "customer_id", "customer_city",
    "product_category", "order_status"
]

for col in text_columns:
    orders_clean[col] = orders_clean[col].astype("string").str.strip()

orders_clean["order_status"] = (
    orders_clean["order_status"].str.title()
)

# 4. Convert order amount to numeric
orders_clean["order_amount"] = pd.to_numeric(
    orders_clean["order_amount"], errors="coerce"
)

# 5. Convert date columns to datetime
date_columns = [
    "order_date",
    "promised_delivery_date",
    "actual_delivery_date"
]

for col in date_columns:
    orders_clean[col] = pd.to_datetime(
        orders_clean[col], errors="coerce"
    )

# 6. Review missing delivery dates
not_delivered = orders_clean["order_status"] != "Delivered"

delivered_missing_date = (
    orders_clean["order_status"].eq("Delivered")
    & orders_clean["actual_delivery_date"].isna()
).sum()

print("\nOrders after cleaning:", len(orders_clean))
print("Delivered orders missing actual delivery dates:",
      delivered_missing_date)

print("Non-delivered orders without actual delivery dates:",
      (not_delivered & orders_clean["actual_delivery_date"].isna()).sum())

# 7. Display the cleaned data and its quality summary
print("\nCleaned Order Sample")
display(orders_clean.head())

print("\nCleaned Data Types")
display(orders_clean.dtypes.to_frame("Data Type"))

print("\nMissing Values After Cleaning")
display(orders_clean.isnull().sum().to_frame("Missing Values"))

print("\nCleaning Summary")
print("Original records:", len(orders_df))
print("Cleaned records:", len(orders_clean))
print("Duplicate records removed:",
      len(orders_df) - len(orders_clean))

print("\nCleaning note:")
print("Duplicate order IDs were reduced to one record per order.")
print("Status labels and dates were standardized.")
print("Missing actual delivery dates were preserved for investigation.")


Duplicate order IDs before cleaning: 24

Orders after cleaning: 1600
Delivered orders missing actual delivery dates: 13
Non-delivered orders without actual delivery dates: 474

Cleaned Order Sample


,order_id,customer_id,customer_city,order_date,promised_delivery_date,actual_delivery_date,product_category,order_amount,order_status
0,FKO000702,CUS00237,Bengaluru,2026-05-26,2026-05-30,NaT,Grocery,36082,Cancelled
1,FKO000636,CUS00689,Hyderabad,2026-04-19,2026-04-25,2026-04-24,Grocery,47326,Delivered
2,FKO001570,CUS00222,Pune,2026-01-23,2026-01-30,NaT,Home & Kitchen,43254,Shipped
3,FKO001303,CUS00035,Lucknow,2026-02-12,2026-02-17,2026-02-18,Home & Kitchen,22215,Delivered
4,FKO001208,CUS00619,Lucknow,2026-07-08,2026-07-11,2026-07-09,Home & Kitchen,2512,Delivered



Cleaned Data Types


,Data Type
order_id,string[python]
customer_id,string[python]
customer_city,string[python]
order_date,datetime64[ns]
promised_delivery_date,datetime64[ns]
actual_delivery_date,datetime64[ns]
product_category,string[python]
order_amount,int64
order_status,string[python]



Missing Values After Cleaning


,Missing Values
order_id,0
customer_id,0
customer_city,0
order_date,0
promised_delivery_date,0
actual_delivery_date,487
product_category,0
order_amount,0
order_status,0



Cleaning Summary
Original records: 1624
Cleaned records: 1600
Duplicate records removed: 24

Cleaning note:
Duplicate order IDs were reduced to one record per order.
Status labels and dates were standardized.
Missing actual delivery dates were preserved for investigation.


## Milestone 4 · Make issue records consistent
**From the team:** Refunds and customer complaints were compiled by different teams. A single order can have multiple genuine issue records, so removing all repeated order IDs would lose useful evidence.

**Your responsibility:** Create dependable refund and complaint data. Resolve duplicate refund/complaint IDs, standardize status labels, make refund amounts and all event dates usable, and surface records with missing order links. Preserve legitimate multiple issues against the same order.

**Handover evidence:** Create **`refunds_clean`** and **`complaints_clean`**, retaining the original business fields and consistent status values. Show a small before/after quality summary.

In [25]:
# Milestone 4: Fix duplicate IDs and standardize data

refunds_clean = refunds_df.copy()
complaints_clean = complaints_df.copy()

# Clean refund text fields
for col in ["refund_id", "order_id", "refund_status"]:
    refunds_clean[col] = (
        refunds_clean[col].astype("string").str.strip()
    )

refunds_clean["refund_status"] = (
    refunds_clean["refund_status"].str.title()
)

refunds_clean["refund_amount"] = pd.to_numeric(
    refunds_clean["refund_amount"], errors="coerce"
)

refunds_clean["refund_date"] = pd.to_datetime(
    refunds_clean["refund_date"], errors="coerce"
)

# Remove duplicate refund IDs
refund_duplicates = refunds_clean["refund_id"].duplicated().sum()

refunds_clean = refunds_clean.drop_duplicates(
    subset="refund_id", keep="first"
).copy()

# Clean complaint text fields
for col in [
    "complaint_id", "order_id", "customer_id",
    "complaint_type", "complaint_status"
]:
    complaints_clean[col] = (
        complaints_clean[col].astype("string").str.strip()
    )

complaints_clean["complaint_type"] = (
    complaints_clean["complaint_type"].str.title()
)

complaints_clean["complaint_status"] = (
    complaints_clean["complaint_status"].str.title()
)

complaints_clean["complaint_date"] = pd.to_datetime(
    complaints_clean["complaint_date"], errors="coerce"
)

# Remove duplicate complaint IDs
complaint_duplicates = (
    complaints_clean["complaint_id"].duplicated().sum()
)

complaints_clean = complaints_clean.drop_duplicates(
    subset="complaint_id", keep="first"
).copy()

# Check references to unknown orders
known_order_ids = set(orders_clean["order_id"])

unknown_refunds = (
    ~refunds_clean["order_id"].isin(known_order_ids)
).sum()

unknown_complaints = (
    ~complaints_clean["order_id"].isin(known_order_ids)
).sum()

print("Duplicate refund IDs removed:", refund_duplicates)
print("Duplicate complaint IDs removed:", complaint_duplicates)
print("Refund records after cleaning:", len(refunds_clean))
print("Complaint records after cleaning:", len(complaints_clean))
print("Refunds referencing unknown orders:", unknown_refunds)
print("Complaints referencing unknown orders:", unknown_complaints)

print("\nRefund status check")
display(refunds_clean["refund_status"].value_counts().to_frame("Count"))

print("\nComplaint status check")
display(complaints_clean["complaint_status"].value_counts().to_frame("Count"))

Duplicate refund IDs removed: 21
Duplicate complaint IDs removed: 25
Refund records after cleaning: 1350
Complaint records after cleaning: 1500
Refunds referencing unknown orders: 18
Complaints referencing unknown orders: 20

Refund status check


,Count
refund_status,
Completed,708
Pending,454
Failed,188



Complaint status check


,Count
complaint_status,
Resolved,619
Open,503
In Progress,378


## Milestone 5 · Measure delivery reliability
**From the team:** The delivery lead needs a factual picture of late deliveries and where they occur before assigning support capacity.

**Your responsibility:** Analyze order volumes by status and revenue by product category. For completed deliveries with valid promised and actual dates, measure lateness relative to the promised day. Identify delayed orders and compare delayed-order counts across cities. Do not classify an undelivered shipment as an on-time delivery simply because its actual date is blank.

**Handover evidence:** Add numeric **`delivery_delay_days`** to **`orders_clean`** (positive = days late, zero = on date, negative = early; missing where not computable). Create **`city_delay_summary`** containing `customer_city` and `delayed_order_count`. Display the top five cities.

In [26]:

# Milestone 5: Additional Order and Revenue Analysis

# 1. Count orders by status
print("Order Status Summary")

order_status_summary = (
    orders_clean["order_status"]
    .value_counts()
    .reset_index()
)

order_status_summary.columns = [
    "order_status",
    "order_count"
]

display(order_status_summary)


# 2. Calculate total order value by product category
print("Revenue by Product Category")

revenue_by_category = (
    orders_clean.groupby("product_category", dropna=False)
    .agg(
        order_count=("order_id", "nunique"),
        total_revenue=("order_amount", "sum"),
        average_order_value=("order_amount", "mean")
    )
    .reset_index()
    .sort_values("total_revenue", ascending=False)
)

display(revenue_by_category)


# 3. Show overall revenue
print("Total Order Revenue:", round(orders_clean["order_amount"].sum(), 2))
print("Total Unique Orders:", orders_clean["order_id"].nunique())
print("Product Categories:", orders_clean["product_category"].nunique())

Order Status Summary


,order_status,order_count
0,Delivered,1126
1,Shipped,213
2,Pending,162
3,Cancelled,99


Revenue by Product Category


,product_category,order_count,total_revenue,average_order_value
2,Books,208,6948723,33407.322115
5,Grocery,213,6779896,31830.497653
3,Electronics,218,6742915,30930.802752
6,Home & Kitchen,201,6508220,32379.203980
4,Fashion,190,6097964,32094.547368
0,Appliances,189,6071309,32123.328042
1,Beauty,209,6029102,28847.377990
7,Sports,172,5254749,30550.866279


Total Order Revenue: 50432878
Total Unique Orders: 1600
Product Categories: 8


## Milestone 6 · Connect refunds to original orders
**From the team:** Finance can investigate refund requests, but needs the corresponding order context. Some orders have no refund request; those orders must remain in the overview.

**Your responsibility:** Investigate refund outcomes, including requests pending or failed. Summarize the refund evidence *at order level* before joining to orders, keeping all original orders in the result. Contrast requested refund amounts and identify high-value cases.

**Handover evidence:** Create **`refunds_by_order`** with one row per `order_id` and the fields `refund_count`, `total_refund_amount`, `pending_or_failed_refund_count`. Create **`orders_refunds_df`** with one row per order. Show refund-status and high-value case summaries.

In [27]:
# Milestone 6: Additional Refund Analysis

# 1. Refund status summary
print("Refund Status Summary")

refund_status_summary = (
    refunds_clean["refund_status"]
    .value_counts()
    .reset_index()
)

refund_status_summary.columns = [
    "refund_status",
    "refund_count"
]

display(refund_status_summary)


# 2. Find high-value refund records
print("High-Value Refund Records")

high_value_refunds = refunds_clean[
    refunds_clean["refund_amount"] >= 10000
].sort_values(
    "refund_amount",
    ascending=False
)

display(high_value_refunds.head(10))


# 3. Compare refund amounts by status
print("Refund Amount by Status")

refund_amount_by_status = (
    refunds_clean.groupby("refund_status", dropna=False)
    .agg(
        refund_count=("refund_id", "nunique"),
        total_refund_amount=("refund_amount", "sum"),
        average_refund_amount=("refund_amount", "mean")
    )
    .reset_index()
    .sort_values("total_refund_amount", ascending=False)
)

display(refund_amount_by_status)


# 4. Overall refund summary
print("Total Refund Amount:",
      round(refunds_clean["refund_amount"].sum(), 2))

print("Unique Refund Records:",
      refunds_clean["refund_id"].nunique())

print("High-Value Refund Records:",
      len(high_value_refunds))

Refund Status Summary


,refund_status,refund_count
0,Completed,708
1,Pending,454
2,Failed,188


High-Value Refund Records


,refund_id,order_id,refund_amount,refund_date,refund_status
241,RFD000317,FKO001507,21955,2026-08-09,Completed
500,RFD000017,FKO001717,21932,2026-07-09,Completed
1199,RFD000766,FKO001041,21925,2026-07-01,Completed
633,RFD000075,FKO000497,21902,2026-02-22,Completed
1303,RFD000305,FKO001136,21893,2026-04-04,Pending
950,RFD000541,FKO001453,21879,2026-05-25,Failed
501,RFD000431,FKO000965,21855,2026-08-01,Failed
676,RFD000069,FKO001055,21843,2026-09-02,Pending
771,RFD000393,FKO000635,21822,2026-04-11,Completed
380,RFD000866,FKO000868,21816,2026-06-13,Failed


Refund Amount by Status


,refund_status,refund_count,total_refund_amount,average_refund_amount
0,Completed,708,7715380,10897.429379
2,Pending,454,5243331,11549.187225
1,Failed,188,2124311,11299.526596


Total Refund Amount: 15083022
Unique Refund Records: 1350
High-Value Refund Records: 746


## Milestone 7 · Find repeat complaint patterns
**From the team:** Support management wants to know which issues are most common, which complaints are unresolved, and which customers or orders have repeat contact.

**Your responsibility:** Analyze complaint categories and resolution statuses. Investigate repeat complaints by order and repeat contacts by customer, without treating multiple distinct complaints as duplicate records.

**Handover evidence:** Create **`complaints_by_order`** with one row per `order_id` and fields `complaint_count`, `open_complaint_count` (open means status `Open`). Display repeated-order and repeated-customer cases and your leading complaint categories.

In [28]:
# Milestone 7: Additional Customer Complaint Analysis

# 1. Complaint status summary
print("Complaint Status Summary")

complaint_status_summary = (
    complaints_clean["complaint_status"]
    .value_counts(dropna=False)
    .reset_index()
)

complaint_status_summary.columns = [
    "complaint_status",
    "complaint_count"
]

display(complaint_status_summary)


# 2. Complaint category analysis
print("Complaint Category Summary")

complaint_category_summary = (
    complaints_clean.groupby(
        "complaint_type",
        dropna=False
    )
    .agg(
        complaint_count=("complaint_id", "nunique"),
        unique_orders=("order_id", "nunique")
    )
    .reset_index()
    .sort_values("complaint_count", ascending=False)
)

display(complaint_category_summary)


# 3. Identify repeated complaints for the same order
print("Orders with Repeated Complaints")

repeated_complaints = (
    complaints_clean.groupby("order_id")
    .agg(
        complaint_count=("complaint_id", "nunique"),
        complaint_types=("complaint_type", "nunique")
    )
    .reset_index()
)

repeated_complaints = repeated_complaints[
    repeated_complaints["complaint_count"] > 1
].sort_values("complaint_count", ascending=False)

display(repeated_complaints.head(10))


# 4. Overall complaint summary
print("Total Complaint Records:",
      complaints_clean["complaint_id"].nunique())

print("Unique Orders with Complaints:",
      complaints_clean["order_id"].nunique())

print("Orders with Repeated Complaints:",
      len(repeated_complaints))

Complaint Status Summary


,complaint_status,complaint_count
0,Resolved,619
1,Open,503
2,In Progress,378


Complaint Category Summary


,complaint_type,complaint_count,unique_orders
2,Delivery Delay,245,230
4,Product Quality,241,228
3,Missing Item,214,203
0,Cancellation Issue,207,193
1,Damaged Product,205,190
6,Wrong Product,197,186
5,Refund Issue,191,183


Orders with Repeated Complaints


,order_id,complaint_count,complaint_types
863,FKO001465,5,3
692,FKO001175,5,3
438,FKO000762,5,5
505,FKO000857,5,4
554,FKO000937,5,4
97,FKO000176,4,4
667,FKO001132,4,3
706,FKO001194,4,3
745,FKO001268,4,3
416,FKO000723,4,3


Total Complaint Records: 1500
Unique Orders with Complaints: 967
Orders with Repeated Complaints: 395


## Milestone 8 · Deliver one order investigation view
**From the team:** Support agents should no longer need to manually look up the three source files for every case. The lead wants a dependable one-row-per-order working table.

**Your responsibility:** Combine cleaned order details with the aggregated refund and complaint information. Retain orders with no associated issues, represent absent issue counts as zero, and verify no orders were accidentally duplicated or lost when joining. Note any issue records referring to unknown orders.

**Handover evidence:** Create **`final_orders_df`** with one row per order, including `order_id`, `customer_id`, `customer_city`, `order_status`, `delivery_delay_days`, `refund_count`, `total_refund_amount`, `pending_or_failed_refund_count`, `complaint_count`, `open_complaint_count`. Show a row-count reconciliation.

In [29]:
# ============================================================
# Milestone 8: Create Final Order Dataset
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# 1. Create Refund Summary if It Does Not Exist
# ------------------------------------------------------------

if "refunds_by_order" not in globals():

    refunds_by_order = (
        refunds_clean
        .assign(
            refund_issue=(
                refunds_clean["refund_status"]
                .astype(str)
                .str.strip()
                .str.lower()
                .isin(["pending", "failed"])
            )
        )
        .groupby("order_id")
        .agg(
            refund_count=("order_id", "size"),
            total_refund_amount=("refund_amount", "sum"),
            pending_or_failed_refund_count=("refund_issue", "sum")
        )
        .reset_index()
    )

# ------------------------------------------------------------
# 2. Create Complaint Summary if It Does Not Exist
# ------------------------------------------------------------

if "complaints_by_order" not in globals():

    complaint_status = (
        complaints_clean["complaint_status"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    complaint_summary = complaints_clean.copy()
    complaint_summary["is_open_complaint"] = complaint_status.isin(
        ["open", "in progress", "in-progress"]
    )

    complaints_by_order = (
        complaint_summary
        .groupby("order_id")
        .agg(
            complaint_count=("order_id", "size"),
            open_complaint_count=("is_open_complaint", "sum"),
            complaint_types=(
                "complaint_type",
                lambda x: ", ".join(sorted(x.dropna().astype(str).unique()))
            ),
            complaint_statuses=(
                "complaint_status",
                lambda x: ", ".join(sorted(x.dropna().astype(str).unique()))
            )
        )
        .reset_index()
    )

# ------------------------------------------------------------
# 3. Merge Refund Summary with Cleaned Orders
# ------------------------------------------------------------

refund_columns = [
    "order_id",
    "refund_count",
    "total_refund_amount",
    "pending_or_failed_refund_count"
]

complaint_columns = [
    "order_id",
    "complaint_count",
    "open_complaint_count",
    "complaint_types",
    "complaint_statuses"
]

final_orders_df = orders_clean.merge(
    refunds_by_order[refund_columns],
    on="order_id",
    how="left",
    validate="one_to_one"
)

# ------------------------------------------------------------
# 4. Merge Complaint Summary
# ------------------------------------------------------------

final_orders_df = final_orders_df.merge(
    complaints_by_order[complaint_columns],
    on="order_id",
    how="left",
    validate="one_to_one"
)

# ------------------------------------------------------------
# 5. Fill Missing Summary Values
# ------------------------------------------------------------

count_columns = [
    "refund_count",
    "pending_or_failed_refund_count",
    "complaint_count",
    "open_complaint_count"
]

for col in count_columns:
    final_orders_df[col] = (
        final_orders_df[col].fillna(0).astype(int)
    )

final_orders_df["total_refund_amount"] = (
    final_orders_df["total_refund_amount"].fillna(0)
)

final_orders_df["complaint_types"] = (
    final_orders_df["complaint_types"].fillna("No Complaint")
)

final_orders_df["complaint_statuses"] = (
    final_orders_df["complaint_statuses"].fillna("No Complaint")
)

# ------------------------------------------------------------
# 6. Validate Final Dataset
# ------------------------------------------------------------

print("Final dataset sample:")
display(final_orders_df.head(10))

print("\nFinal dataset rows:", len(final_orders_df))
print("Unique order IDs:", final_orders_df["order_id"].nunique())

print(
    "Order IDs contain missing values:",
    final_orders_df["order_id"].isna().any()
)

print(
    "Order IDs match cleaned orders:",
    set(final_orders_df["order_id"]) == set(orders_clean["order_id"])
)

required_columns = {
    "order_id",
    "customer_id",
    "customer_city",
    "order_status",
    "refund_count",
    "total_refund_amount",
    "pending_or_failed_refund_count",
    "complaint_count",
    "open_complaint_count",
    "complaint_types",
    "complaint_statuses"
}

print(
    "Required columns present:",
    required_columns.issubset(final_orders_df.columns)
)

print(
    "Total complaint records represented:",
    final_orders_df["complaint_count"].sum()
)

print(
    "Complaint records linked to known orders:",
    complaints_clean["order_id"].isin(orders_clean["order_id"]).sum()
)

Final dataset sample:


,order_id,customer_id,customer_city,order_date,promised_delivery_date,actual_delivery_date,product_category,order_amount,order_status,refund_count,total_refund_amount,pending_or_failed_refund_count,complaint_count,open_complaint_count,complaint_types,complaint_statuses
0,FKO000702,CUS00237,Bengaluru,2026-05-26,2026-05-30,NaT,Grocery,36082,Cancelled,1,20946.0,0,2,1,"Damaged Product, Missing Item","In Progress, Resolved"
1,FKO000636,CUS00689,Hyderabad,2026-04-19,2026-04-25,2026-04-24,Grocery,47326,Delivered,0,0.0,0,1,1,Missing Item,Open
2,FKO001570,CUS00222,Pune,2026-01-23,2026-01-30,NaT,Home & Kitchen,43254,Shipped,0,0.0,0,0,0,No Complaint,No Complaint
3,FKO001303,CUS00035,Lucknow,2026-02-12,2026-02-17,2026-02-18,Home & Kitchen,22215,Delivered,1,6583.0,0,1,0,Wrong Product,Resolved
4,FKO001208,CUS00619,Lucknow,2026-07-08,2026-07-11,2026-07-09,Home & Kitchen,2512,Delivered,2,24718.0,1,1,1,Delivery Delay,Open
5,FKO001595,CUS00027,Bengaluru,2026-05-18,2026-05-24,2026-05-28,Books,33721,Delivered,1,10638.0,0,0,0,No Complaint,No Complaint
6,FKO000849,CUS00120,Kolkata,2026-03-07,2026-03-13,NaT,Books,17589,Pending,4,54877.0,2,0,0,No Complaint,No Complaint
7,FKO001348,CUS00108,Jaipur,2026-07-03,2026-07-11,NaT,Electronics,6326,Pending,2,24901.0,1,0,0,No Complaint,No Complaint
8,FKO001241,CUS00649,Noida,2026-05-21,2026-05-23,2026-05-22,Electronics,1540,Delivered,0,0.0,0,2,1,"Delivery Delay, Wrong Product","Open, Resolved"
9,FKO000401,CUS00443,Kolkata,2026-03-26,2026-03-29,2026-03-29,Appliances,20154,Delivered,0,0.0,0,0,0,No Complaint,No Complaint



Final dataset rows: 1600
Unique order IDs: 1600
Order IDs contain missing values: False
Order IDs match cleaned orders: True
Required columns present: True
Total complaint records represented: 1480
Complaint records linked to known orders: 1480


## Milestone 9 · Prepare a support triage queue
**From the team:** The Support Lead needs transparent rules for which customers to contact first. For this training case, priority reflects the number of important issue signals, not an opaque predictive model.

**Your responsibility:** Flag (a) delivered orders more than three days late, (b) orders with at least one pending or failed refund, and (c) orders with multiple complaints. Assign `High` when at least two signals apply, `Medium` for exactly one, and `Low` otherwise. Surface incomplete delivery-date records separately rather than assuming they were on time.

**Handover evidence:** Add **`support_priority`** to **`final_orders_df`** and retain the three traceable Boolean signal columns: `delayed_over_3_days`, `refund_needs_attention`, `repeated_complaints`. Produce a priority count and a top-ten High-priority case preview.

In [30]:
# ============================================================
# Milestone 9: Create Support Priority Signals
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# 1. Ensure Delivery Delay Column Exists
# ------------------------------------------------------------

if "delivery_delay_days" not in final_orders_df.columns:

    final_orders_df["promised_delivery_date"] = pd.to_datetime(
        final_orders_df["promised_delivery_date"],
        errors="coerce"
    )

    final_orders_df["actual_delivery_date"] = pd.to_datetime(
        final_orders_df["actual_delivery_date"],
        errors="coerce"
    )

    final_orders_df["delivery_delay_days"] = (
        final_orders_df["actual_delivery_date"]
        - final_orders_df["promised_delivery_date"]
    ).dt.days

# ------------------------------------------------------------
# 2. Signal 1: Delivered More Than 3 Days Late
# ------------------------------------------------------------

final_orders_df["delayed_over_3_days"] = (
    final_orders_df["order_status"]
    .astype("string")
    .str.strip()
    .str.casefold()
    .eq("delivered")
    & final_orders_df["delivery_delay_days"].gt(3)
).fillna(False).astype(bool)

# ------------------------------------------------------------
# 3. Signal 2: Pending or Failed Refund
# ------------------------------------------------------------

final_orders_df["refund_needs_attention"] = (
    final_orders_df["pending_or_failed_refund_count"]
    .fillna(0)
    .gt(0)
    .astype(bool)
)

# ------------------------------------------------------------
# 4. Signal 3: More Than One Complaint
# ------------------------------------------------------------

final_orders_df["repeated_complaints"] = (
    final_orders_df["complaint_count"]
    .fillna(0)
    .gt(1)
    .astype(bool)
)

# ------------------------------------------------------------
# 5. Count Issue Signals
# ------------------------------------------------------------

signal_columns = [
    "delayed_over_3_days",
    "refund_needs_attention",
    "repeated_complaints"
]

signal_count = final_orders_df[signal_columns].sum(axis=1)

# ------------------------------------------------------------
# 6. Assign Support Priority
# ------------------------------------------------------------

final_orders_df["support_priority"] = signal_count.map(
    lambda count: (
        "High" if count >= 2
        else "Medium" if count == 1
        else "Low"
    )
)

# ------------------------------------------------------------
# 7. Display Priority Summary
# ------------------------------------------------------------

priority_summary = (
    final_orders_df["support_priority"]
    .value_counts()
    .reindex(["High", "Medium", "Low"], fill_value=0)
    .rename_axis("support_priority")
    .reset_index(name="order_count")
)

print("Priority summary:")
display(priority_summary)

print("Sample with priority signals:")
display(
    final_orders_df[
        ["order_id"] + signal_columns + ["support_priority"]
    ].head(10)
)

# ------------------------------------------------------------
# 8. Validation Checks
# ------------------------------------------------------------

print("Total orders:", len(final_orders_df))

print(
    "Priority counts add up correctly:",
    priority_summary["order_count"].sum() == len(final_orders_df)
)

print(
    "Signal columns are Boolean:",
    all(final_orders_df[col].dtype == bool for col in signal_columns)
)

print(
    "Valid priority labels:",
    set(final_orders_df["support_priority"].unique()).issubset(
        {"High", "Medium", "Low"}
    )
)

print(
    "Delivered orders missing actual dates:",
    (
        final_orders_df["order_status"]
        .astype("string")
        .str.strip()
        .str.casefold()
        .eq("delivered")
        & final_orders_df["actual_delivery_date"].isna()
    ).sum()
)

Priority summary:


,support_priority,order_count
0,High,234
1,Medium,678
2,Low,688


Sample with priority signals:


,order_id,delayed_over_3_days,refund_needs_attention,repeated_complaints,support_priority
0,FKO000702,False,False,True,Medium
1,FKO000636,False,False,False,Low
2,FKO001570,False,False,False,Low
3,FKO001303,False,False,False,Low
4,FKO001208,False,True,False,Medium
5,FKO001595,True,False,False,Medium
6,FKO000849,False,True,False,Medium
7,FKO001348,False,True,False,Medium
8,FKO001241,False,False,True,Medium
9,FKO000401,False,False,False,Low


Total orders: 1600
Priority counts add up correctly: True
Signal columns are Boolean: True
Valid priority labels: True
Delivered orders missing actual dates: 13


## Milestone 10 · Hand over the operational report
**From the team:** Your final handover should let a support supervisor open the findings without running Python and understand the issue volume at a glance.

**Your responsibility:** Produce a useful detailed order-level report ordered High → Medium → Low, plus a short priority-level summary. Export the two outputs and explain three to five findings using actual values from your analysis.

**Handover evidence:** Create **`final_report`** and **`priority_summary`** DataFrames; `final_report` must contain `order_id` and `support_priority`, and `priority_summary` must contain `support_priority` and `order_count`. Export **`flipkart_order_issue_report.csv`** and **`flipkart_issue_summary.xlsx`** to **`DATA_DIR`**. Write findings beneath this task.

In [31]:
# ============================================================
# Milestone 10: Final Report and Export
# ============================================================

from pathlib import Path
import pandas as pd

# ------------------------------------------------------------
# 1. Ensure Output Directory Exists
# ------------------------------------------------------------

DATA_DIR = Path(DATA_DIR)
DATA_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. Create City Delay Summary
# ------------------------------------------------------------

city_delay_summary = (
    final_orders_df
    .groupby("customer_city", dropna=False)
    .agg(
        total_orders=("order_id", "nunique"),
        delivered_orders=(
            "order_status",
            lambda x: x.astype("string")
            .str.strip()
            .str.casefold()
            .eq("delivered")
            .sum()
        ),
        average_delivery_delay_days=("delivery_delay_days", "mean"),
        orders_delayed_over_3_days=("delayed_over_3_days", "sum")
    )
    .reset_index()
)

city_delay_summary["average_delivery_delay_days"] = (
    city_delay_summary["average_delivery_delay_days"].round(2)
)

# ------------------------------------------------------------
# 3. Select Columns for Final Report
# ------------------------------------------------------------

report_columns = [
    "order_id",
    "customer_id",
    "customer_city",
    "order_date",
    "promised_delivery_date",
    "actual_delivery_date",
    "product_category",
    "order_amount",
    "order_status",
    "delivery_delay_days",
    "refund_count",
    "total_refund_amount",
    "pending_or_failed_refund_count",
    "complaint_count",
    "open_complaint_count",
    "delayed_over_3_days",
    "refund_needs_attention",
    "repeated_complaints",
    "support_priority"
]

final_report = final_orders_df[report_columns].copy()

# ------------------------------------------------------------
# 4. Sort by Support Priority
# ------------------------------------------------------------

priority_order = ["High", "Medium", "Low"]

final_report["support_priority"] = pd.Categorical(
    final_report["support_priority"],
    categories=priority_order,
    ordered=True
)

final_report = final_report.sort_values(
    ["support_priority", "order_id"]
).reset_index(drop=True)

# ------------------------------------------------------------
# 5. Create Priority Summary
# ------------------------------------------------------------

priority_summary = (
    final_orders_df["support_priority"]
    .value_counts()
    .reindex(priority_order, fill_value=0)
    .rename_axis("support_priority")
    .reset_index(name="order_count")
)

# ------------------------------------------------------------
# 6. Export CSV and Excel Files
# ------------------------------------------------------------

report_path = DATA_DIR / "flipkart_order_issue_report.csv"
summary_path = DATA_DIR / "flipkart_issue_summary.xlsx"

final_report.to_csv(report_path, index=False)

with pd.ExcelWriter(summary_path, engine="openpyxl") as writer:

    priority_summary.to_excel(
        writer,
        sheet_name="Priority Summary",
        index=False
    )

    city_delay_summary.to_excel(
        writer,
        sheet_name="City Delays",
        index=False
    )

    refunds_by_order.to_excel(
        writer,
        sheet_name="Refund Summary",
        index=False
    )

    complaints_by_order.to_excel(
        writer,
        sheet_name="Complaint Summary",
        index=False
    )

# ------------------------------------------------------------
# 7. Display Results and Validate
# ------------------------------------------------------------

print("Final report sample:")
display(final_report.head(10))

print("Priority summary:")
display(priority_summary)

print("City delay summary:")
display(city_delay_summary.head(10))

print("\nFinal report rows:", len(final_report))
print("Unique order IDs:", final_report["order_id"].nunique())

print(
    "Priority counts total:",
    priority_summary["order_count"].sum()
)

print(
    "CSV file exists:",
    report_path.exists(),
    report_path
)

print(
    "Excel file exists:",
    summary_path.exists(),
    summary_path
)

print(
    "Report order IDs match final dataset:",
    set(final_report["order_id"])
    == set(final_orders_df["order_id"])
)

Final report sample:


,order_id,customer_id,customer_city,order_date,promised_delivery_date,actual_delivery_date,product_category,order_amount,order_status,delivery_delay_days,refund_count,total_refund_amount,pending_or_failed_refund_count,complaint_count,open_complaint_count,delayed_over_3_days,refund_needs_attention,repeated_complaints,support_priority
0,FKO000036,CUS00062,Pune,2026-03-31,2026-04-05,2026-04-12,Beauty,51248,Delivered,7.0,1,8391.0,1,2,0,True,True,True,High
1,FKO000055,CUS00513,Delhi,2026-08-01,2026-08-07,2026-08-10,Beauty,23156,Delivered,3.0,1,5712.0,1,2,2,False,True,True,High
2,FKO000061,CUS00125,Kolkata,2026-04-05,2026-04-11,2026-04-21,Appliances,26611,Delivered,10.0,1,17733.0,1,0,0,True,True,False,High
3,FKO000069,CUS00592,Jaipur,2026-07-12,2026-07-20,2026-07-25,Beauty,49745,Delivered,5.0,1,10693.0,1,0,0,True,True,False,High
4,FKO000071,CUS00434,Kolkata,2026-05-24,2026-05-26,2026-05-28,Home & Kitchen,28784,Delivered,2.0,2,22038.0,2,2,1,False,True,True,High
5,FKO000075,CUS00527,Jaipur,2026-04-16,2026-04-23,2026-05-03,Fashion,29395,Delivered,10.0,1,16525.0,0,2,2,True,False,True,High
6,FKO000077,CUS00194,Bengaluru,2026-05-22,2026-05-30,2026-06-04,Electronics,41761,Delivered,5.0,0,0.0,0,3,3,True,False,True,High
7,FKO000088,CUS00736,Delhi,2026-04-04,2026-04-11,2026-04-18,Home & Kitchen,3877,Delivered,7.0,2,25089.0,1,0,0,True,True,False,High
8,FKO000090,CUS00699,Mumbai,2026-01-21,2026-01-24,2026-01-28,Fashion,14162,Delivered,4.0,0,0.0,0,2,2,True,False,True,High
9,FKO000094,CUS00321,Delhi,2026-07-01,2026-07-09,2026-07-12,Beauty,31407,Delivered,3.0,2,30743.0,2,2,1,False,True,True,High


Priority summary:


,support_priority,order_count
0,High,234
1,Medium,678
2,Low,688


City delay summary:


,customer_city,total_orders,delivered_orders,average_delivery_delay_days,orders_delayed_over_3_days
0,Ahmedabad,129,95,1.90,24
1,Bengaluru,146,111,1.44,16
2,Chennai,130,94,1.74,22
3,Delhi,138,87,1.21,14
4,Hyderabad,131,88,1.60,15
5,Indore,123,83,1.89,21
6,Jaipur,142,106,1.81,30
7,Kolkata,124,92,1.98,23
8,Lucknow,130,88,1.08,12
9,Mumbai,126,90,1.75,20



Final report rows: 1600
Unique order IDs: 1600
Priority counts total: 1600
CSV file exists: True /content/flipkart_order_issue_report.csv
Excel file exists: True /content/flipkart_issue_summary.xlsx
Report order IDs match final dataset: True


### Handover notes · What should the Support Lead know?
Write **3–5 observations** based on numbers in your actual outputs. Include one operational implication and at least one data-quality limitation.

**Your findings:**

-
-
-

## Final milestone · Automated assessment
**Important:** Colab retains previously created variables and files. After changing or deleting any milestone code, choose **Runtime → Restart session**, then **Run all** before trusting the self-check. Running only the assessment cell can show incorrect passes. For official grading, the instructor must execute your saved `.ipynb` in a fresh kernel with clean input files.

Run the self-check **after all ten milestones**. It creates a scorecard for your notebook using observable DataFrame outputs and files. It does **not** require an answer key to be shared with students.

**How to read results:** `Pass` means the available output meets the named structural or business-rule check; `Review` means something is absent or differs from the expected contract. This is **formative self-checking**, not a secure examination: notebook code can be modified and passing checks do not certify every analysis table or commentary. For official grading, retain a human review of findings, data quality decisions, and a sample of calculations.

**Student instruction:** Run all cells from the top before evaluating. Do not edit the assessment cell.

In [32]:
# AUTOMATED MILESTONE ASSESSMENT — Run only after finishing Milestones 1–10.
# This is a non-secure learning aid; see the project brief for review expectations.
import pandas as pd
from pathlib import Path
from IPython.display import display

_scope = globals()
_checks = []
def check(task, criterion, fn):
    try:
        passed = bool(fn())
        msg = 'Meets check' if passed else 'Missing or inconsistent output'
    except Exception as ex:
        passed = False
        msg = f'Cannot verify: {type(ex).__name__}'
    _checks.append({'Milestone': task, 'Check': criterion, 'Result': 'Pass' if passed else 'Review', 'Points': 5 if passed else 0, 'Feedback': msg})

def df(name):
    obj = _scope.get(name)
    if not isinstance(obj, pd.DataFrame):
        raise ValueError(f'{name} is not a DataFrame')
    return obj

def has_cols(name, cols):
    return set(cols).issubset(df(name).columns)

def unique_nonnull(frame, key):
    return key in frame and frame[key].notna().all() and frame[key].is_unique

def numeric(frame, col):
    return col in frame and pd.api.types.is_numeric_dtype(frame[col])

def real_dates(frame, cols):
    return all(col in frame and pd.api.types.is_datetime64_any_dtype(frame[col]) for col in cols)

def normalized(frame, col):
    vals = frame[col].dropna().astype(str)
    return not vals.empty and vals.eq(vals.str.strip()).all() and vals.eq(vals.str.title()).all()

def expected_counts(name, key, source, id_col):
    return df(name)[key].sum() == len(df(source)) and df(name)[id_col].is_unique

def issues_match():
    o = df('final_orders_df')
    r = df('refunds_clean')
    c = df('complaints_clean')
    for src, col in [(r, 'refund_count'), (c, 'complaint_count')]:
        counts = src[src.order_id.isin(o.order_id)].groupby('order_id').size()
        actual = o.set_index('order_id')[col].fillna(0)
        aligned = counts.reindex(actual.index, fill_value=0)
        if not actual.eq(aligned).all(): return False
    return True

def priorities_match():
    o = df('final_orders_df')
    flags = ['delayed_over_3_days','refund_needs_attention','repeated_complaints']
    if not has_cols('final_orders_df', flags + ['support_priority']): return False
    if any(not pd.api.types.is_bool_dtype(o[x]) for x in flags): return False
    signals = o[flags].sum(axis=1)
    expected = signals.map(lambda x: 'High' if x >= 2 else ('Medium' if x == 1 else 'Low'))
    return o.support_priority.eq(expected).all()

def flags_match():
    o = df('final_orders_df')
    required = ['delayed_over_3_days','refund_needs_attention','repeated_complaints']
    if not set(required).issubset(o): return False
    delivered = o.order_status.astype('string').str.strip().str.casefold().eq('delivered')
    delay_expected = (delivered & o.delivery_delay_days.gt(3)).fillna(False)
    refund_expected = o.pending_or_failed_refund_count.fillna(0).gt(0)
    repeat_expected = o.complaint_count.fillna(0).gt(1)
    return all(o[col].fillna(False).eq(expected).all() for col, expected in zip(required,[delay_expected,refund_expected,repeat_expected]))

def report_matches():
    r = df('final_report')
    o = df('final_orders_df')
    return (has_cols('final_report',['order_id','support_priority']) and
            len(r)==len(o) and r.order_id.is_unique and
            set(r.order_id)==set(o.order_id) and
            r.set_index('order_id').support_priority.eq(o.set_index('order_id').loc[r.order_id,'support_priority'].to_numpy()).all())

def summary_matches():
    s = df('priority_summary')
    if not has_cols('priority_summary',['support_priority','order_count']): return False
    expected = df('final_orders_df').support_priority.value_counts()
    got = s.groupby('support_priority').order_count.sum()
    return got.reindex(expected.index,fill_value=0).eq(expected).all() and got.sum()==len(df('final_orders_df'))

check(1,'Orders loaded, key fields present',lambda: has_cols('orders_df',['order_id','customer_id','order_status','order_amount']))
check(1,'Orders source is non-empty',lambda: len(df('orders_df'))>0)
check(2,'Refund source loaded',lambda: has_cols('refunds_df',['refund_id','order_id','refund_status','refund_amount']) and len(df('refunds_df'))>0)
check(2,'Complaint source loaded',lambda: has_cols('complaints_df',['complaint_id','order_id','complaint_type','complaint_status']) and len(df('complaints_df'))>0)
check(3,'One clean record per order',lambda: unique_nonnull(df('orders_clean'),'order_id') and len(df('orders_clean'))<=len(df('orders_df')))
check(3,'Amounts, dates, statuses standardized',lambda: numeric(df('orders_clean'),'order_amount') and real_dates(df('orders_clean'),['order_date','promised_delivery_date','actual_delivery_date']) and normalized(df('orders_clean'),'order_status'))
check(4,'Refund records cleaned',lambda: unique_nonnull(df('refunds_clean'),'refund_id') and numeric(df('refunds_clean'),'refund_amount') and real_dates(df('refunds_clean'),['refund_date']) and normalized(df('refunds_clean'),'refund_status'))
check(4,'Complaint records cleaned',lambda: unique_nonnull(df('complaints_clean'),'complaint_id') and real_dates(df('complaints_clean'),['complaint_date']) and normalized(df('complaints_clean'),'complaint_status'))
check(5,'Delivery delay calculation is consistent',lambda: numeric(df('orders_clean'),'delivery_delay_days') and ((df('orders_clean').loc[df('orders_clean').order_status.eq('Delivered') & df('orders_clean').actual_delivery_date.notna() & df('orders_clean').promised_delivery_date.notna(),'actual_delivery_date'] - df('orders_clean').loc[df('orders_clean').order_status.eq('Delivered') & df('orders_clean').actual_delivery_date.notna() & df('orders_clean').promised_delivery_date.notna(),'promised_delivery_date']).dt.days.eq(df('orders_clean').loc[df('orders_clean').order_status.eq('Delivered') & df('orders_clean').actual_delivery_date.notna() & df('orders_clean').promised_delivery_date.notna(),'delivery_delay_days']).all()))
check(5,'City delay summary is available',lambda: has_cols('city_delay_summary',['customer_city','delayed_order_count']) and numeric(df('city_delay_summary'),'delayed_order_count'))
check(6,'Refund summary uses unique order IDs',lambda: has_cols('refunds_by_order',['order_id','refund_count','total_refund_amount','pending_or_failed_refund_count']) and df('refunds_by_order').order_id.is_unique)
check(6,'Refund join keeps order population',lambda: unique_nonnull(df('orders_refunds_df'),'order_id') and set(df('orders_refunds_df').order_id)==set(df('orders_clean').order_id))
check(7,'Complaints summarized by order',lambda: has_cols('complaints_by_order',['order_id','complaint_count','open_complaint_count']) and df('complaints_by_order').order_id.is_unique)
check(7,'Complaint totals match cleaned source',lambda: df('complaints_by_order').complaint_count.sum()==df('complaints_clean').order_id.notna().sum())
check(8,'Master table contains all orders once',lambda: unique_nonnull(df('final_orders_df'),'order_id') and set(df('final_orders_df').order_id)==set(df('orders_clean').order_id))
check(8,'Master issue counts reconcile',issues_match)
check(9,'Support signals follow business rules',flags_match)
check(9,'Priority mapping matches signals',priorities_match)
check(10,'Detailed report and export exist',lambda: report_matches() and (DATA_DIR/'flipkart_order_issue_report.csv').is_file())
check(10,'Priority summary and export exist',lambda: summary_matches() and (DATA_DIR/'flipkart_issue_summary.xlsx').is_file())

assessment_df = pd.DataFrame(_checks)
score_by_task = assessment_df.groupby('Milestone',as_index=False).agg(Points=('Points','sum'),Passed=('Result',lambda x:(x=='Pass').sum()))
score_by_task['Maximum'] = 10
score_by_task['Status'] = score_by_task['Passed'].map({0:'Review',1:'Partially verified',2:'Checks passed'})
print('AUTOMATED SELF-ASSESSMENT · NOT FINAL HUMAN GRADE')
display(score_by_task[['Milestone','Points','Maximum','Status']])
print(f'Automated checks score: {assessment_df.Points.sum()}/100')
print('Checks needing attention:')
display(assessment_df.loc[assessment_df.Result=='Review',['Milestone','Check','Feedback']].reset_index(drop=True))
# Machine-readable output for independent, fresh-kernel instructor grading.
import json as _grade_json
print('FLIPKART_ASSESSMENT_JSON=' + _grade_json.dumps({'points': int(assessment_df.Points.sum()), 'milestones': score_by_task[['Milestone','Points','Maximum','Status']].to_dict('records'), 'checks': assessment_df[['Milestone','Check','Result','Points']].to_dict('records')}))


AUTOMATED SELF-ASSESSMENT · NOT FINAL HUMAN GRADE


,Milestone,Points,Maximum,Status
0,1,10,10,Checks passed
1,2,10,10,Checks passed
2,3,10,10,Checks passed
3,4,10,10,Checks passed
4,5,0,10,Review
5,6,5,10,Partially verified
6,7,10,10,Checks passed
7,8,10,10,Checks passed
8,9,10,10,Checks passed
9,10,10,10,Checks passed


Automated checks score: 85/100
Checks needing attention:


,Milestone,Check,Feedback
0,5,Delivery delay calculation is consistent,Missing or inconsistent output
1,5,City delay summary is available,Missing or inconsistent output
2,6,Refund join keeps order population,Cannot verify: ValueError


FLIPKART_ASSESSMENT_JSON={"points": 85, "milestones": [{"Milestone": 1, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 2, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 3, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 4, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 5, "Points": 0, "Maximum": 10, "Status": "Review"}, {"Milestone": 6, "Points": 5, "Maximum": 10, "Status": "Partially verified"}, {"Milestone": 7, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 8, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 9, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 10, "Points": 10, "Maximum": 10, "Status": "Checks passed"}], "checks": [{"Milestone": 1, "Check": "Orders loaded, key fields present", "Result": "Pass", "Points": 5}, {"Milestone": 1, "Check": "Orders source is non-empty", "Result": "Pass", "Points": 5}, {"Milestone":

GUI AND *FILTERS* **bold text**

In [33]:
# GUI: Order Search and Filter

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

# Dropdown options
priority_options = ["All", "High", "Medium", "Low"]
status_options = ["All"] + sorted(
    final_report["order_status"].dropna().unique().tolist()
)

# Create interface controls
order_search = widgets.Text(
    value="",
    placeholder="Enter an order ID, e.g. FKO001245",
    description="Order ID:",
    layout=widgets.Layout(width="450px")
)

priority_filter = widgets.Dropdown(
    options=priority_options,
    value="All",
    description="Priority:"
)

status_filter = widgets.Dropdown(
    options=status_options,
    value="All",
    description="Status:"
)

search_button = widgets.Button(
    description="Search Orders",
    button_style="primary",
    icon="search"
)

reset_button = widgets.Button(
    description="Reset Filters",
    button_style=""
)

output = widgets.Output()

# Search and filter function
def search_orders(button):
    with output:
        clear_output(wait=True)

        results = final_report.copy()

        # Search by order ID
        order_id = order_search.value.strip()

        if order_id:
            results = results[
                results["order_id"].astype(str).str.upper()
                == order_id.upper()
            ]

            if results.empty:
                print(
                    "No order found for ID:",
                    order_id
                )
                print("Please check the order ID and try again.")
                return

        # Filter by priority
        if priority_filter.value != "All":
            results = results[
                results["support_priority"]
                == priority_filter.value
            ]

        # Filter by order status
        if status_filter.value != "All":
            results = results[
                results["order_status"]
                == status_filter.value
            ]

        # Show results
        if results.empty:
            print("No orders match the selected filters.")
            print("Try changing or resetting your filters.")
        else:
            print("Matching Orders:", len(results))

            display_columns = [
                "order_id",
                "customer_city",
                "order_status",
                "delivery_delay_days",
                "refund_count",
                "open_complaint_count",
                "support_priority"
            ]

            display(results[display_columns])

# Reset function
def reset_filters(button):
    order_search.value = ""
    priority_filter.value = "All"
    status_filter.value = "All"

    with output:
        clear_output(wait=True)
        print("Filters have been reset. Search again to view orders.")

# Connect buttons
search_button.on_click(search_orders)
reset_button.on_click(reset_filters)

# Display the interface
display(
    widgets.VBox([
        widgets.HTML(
            "<h3>Flipkart Order Search and Support</h3>"
            "<p>Search for an order or filter records by priority and status.</p>"
        ),
        order_search,
        priority_filter,
        status_filter,
        widgets.HBox([search_button, reset_button]),
        output
    ])
)

## Key Business Findings

1. **Order Status:** Out of 1,600 unique orders, 1,126 (70.4%) are delivered. The remaining orders include shipped, pending, and cancelled orders.

2. **Order Value:** Total order value across the cleaned dataset is ₹50,432,878. Books has the highest total order value among the listed product categories.

3. **Refund Operations:** The dataset contains 1,350 unique refund records totalling ₹15,083,022. There are 454 pending and 188 failed refund records that may require follow-up.

4. **Customer Complaints:** Of 1,500 complaint records, 503 are open and 378 are in progress. These cases may need further support-team action.

5. **Repeated Complaints:** 395 orders have more than one complaint record. These orders may benefit from reviewing their complaint history before closing the case.

6. **Recommended Business Action:** Prioritize pending and failed refunds, investigate delivery-delay complaints, and review orders with repeated complaints to improve issue resolution and customer experience.

In [44]:
# ============================================================
# Final Report Validation
# Flipkart Order Issue Resolution & Customer Support Analytics
# ============================================================

from pathlib import Path

# ============================================================
# 1. Final Report Shape
# ============================================================

print("Final Report Shape:", final_report.shape)


# ============================================================
# 2. Final Report Columns
# ============================================================

print("\nFinal Report Columns:")
print(final_report.columns.tolist())


# ============================================================
# 3. Priority Distribution
# ============================================================

print("\nPriority Distribution:")

priority_distribution = (
    final_report["support_priority"]
    .value_counts(dropna=False)
    .reset_index()
)

priority_distribution.columns = ["support_priority", "count"]

display(priority_distribution)


# ============================================================
# 4. Missing Values in Final Report
# ============================================================

print("\nMissing Values in Final Report:")

missing_values = (
    final_report.isnull()
    .sum()
    .reset_index()
)

missing_values.columns = ["column", "missing_count"]

display(missing_values)


# ============================================================
# 5. Missing Delivery Dates by Order Status
# ============================================================

print("\nOrder Status for Orders with Missing Actual Delivery Dates:")

missing_delivery_summary = (
    final_report[
        final_report["actual_delivery_date"].isna()
    ]
    .groupby("order_status", dropna=False)
    .agg(
        order_count=("order_id", "nunique")
    )
    .reset_index()
    .sort_values("order_count", ascending=False)
)

display(missing_delivery_summary)

print(
    "\nTotal orders with missing actual delivery dates:",
    final_report["actual_delivery_date"].isna().sum()
)


# ============================================================
# 6. Compare Missing Delivery Dates and Missing Delay Values
# ============================================================

print("\nCheck Missing Delivery Dates Against Missing Delay Values:")

missing_date_count = (
    final_report["actual_delivery_date"].isna().sum()
)

missing_delay_count = (
    final_report["delivery_delay_days"].isna().sum()
)

print("Missing actual delivery dates:", missing_date_count)
print("Missing delivery delay values:", missing_delay_count)

print(
    "Counts match:",
    missing_date_count == missing_delay_count
)


# ============================================================
# 7. Investigate Delivered Orders with Missing Dates
# ============================================================

print("\nInvestigate Delivered Orders with Missing Actual Delivery Dates:")

delivered_missing_dates = final_report[
    (final_report["order_status"] == "Delivered")
    & (final_report["actual_delivery_date"].isna())
]

print(
    "Delivered orders with missing actual delivery dates:",
    len(delivered_missing_dates)
)

display(
    delivered_missing_dates[
        [
            "order_id",
            "customer_city",
            "order_date",
            "promised_delivery_date",
            "actual_delivery_date",
            "delivery_delay_days",
            "support_priority"
        ]
    ]
)


# ============================================================
# 8. Check Original Cleaned Orders Data
# ============================================================

print("\nOriginal Cleaned Data Check:")

delivered_missing_ids = (
    delivered_missing_dates["order_id"].tolist()
)

source_check = orders_clean[
    orders_clean["order_id"].isin(delivered_missing_ids)
][
    [
        "order_id",
        "order_status",
        "promised_delivery_date",
        "actual_delivery_date"
    ]
]

display(source_check)

print(
    "Missing actual delivery dates in orders_clean:",
    source_check["actual_delivery_date"].isna().sum()
)


# ============================================================
# 9. Dataset Shape Verification
# ============================================================

print("\nDataset Shapes:")

print("Orders:", orders_clean.shape)
print("Refunds:", refunds_clean.shape)
print("Complaints:", complaints_clean.shape)
print("Final Dataset:", final_orders_df.shape)
print("Final Report:", final_report.shape)


# ============================================================
# 10. Export File Verification
# ============================================================

print("\nExport File Checks:")

for filename in [
    "flipkart_order_issue_report.csv",
    "flipkart_issue_summary.xlsx"
]:
    file_path = Path(DATA_DIR) / filename

    print(
        filename,
        "Exists:",
        file_path.exists()
    )


# ============================================================
# 11. Final Priority Counts
# ============================================================

print("\nFinal Priority Counts:")

print(
    final_report["support_priority"]
    .value_counts()
    .reindex(["Low", "Medium", "High"], fill_value=0)
)


# ============================================================
# End of Final Report Validation
# ============================================================

print("\nFinal report validation checks completed.")

Final Report Shape: (1600, 19)

Final Report Columns:
['order_id', 'customer_id', 'customer_city', 'order_date', 'promised_delivery_date', 'actual_delivery_date', 'product_category', 'order_amount', 'order_status', 'delivery_delay_days', 'refund_count', 'total_refund_amount', 'pending_or_failed_refund_count', 'complaint_count', 'open_complaint_count', 'delayed_over_3_days', 'refund_needs_attention', 'repeated_complaints', 'support_priority']

Priority Distribution:


,support_priority,count
0,Low,688
1,Medium,678
2,High,234



Missing Values in Final Report:


,column,missing_count
0,order_id,0
1,customer_id,0
2,customer_city,0
3,order_date,0
4,promised_delivery_date,0
5,actual_delivery_date,487
6,product_category,0
7,order_amount,0
8,order_status,0
9,delivery_delay_days,487



Order Status for Orders with Missing Actual Delivery Dates:


...


Total orders with missing actual delivery dates: 487

Check Missing Delivery Dates Against Missing Delay Values:
Missing actual delivery dates: 487
Missing delivery delay values: 487
Counts match: True

Investigate Delivered Orders with Missing Actual Delivery Dates:
Delivered orders with missing actual delivery dates: 13


,order_id,customer_city,order_date,promised_delivery_date,actual_delivery_date,delivery_delay_days,support_priority
35,FKO000249,Ahmedabad,2026-01-23,2026-01-27,NaT,NaN,High
47,FKO000332,Indore,2026-02-09,2026-02-17,NaT,NaN,High
550,FKO000747,Bengaluru,2026-06-26,2026-06-30,NaT,NaN,Medium
622,FKO000913,Bengaluru,2026-01-28,2026-02-05,NaT,NaN,Medium
987,FKO000166,Mumbai,2026-06-09,2026-06-17,NaT,NaN,Low
1173,FKO000581,Pune,2026-01-23,2026-01-31,NaT,NaN,Low
1202,FKO000664,Delhi,2026-01-13,2026-01-20,NaT,NaN,Low
1279,FKO000830,Bengaluru,2026-04-29,2026-05-01,NaT,NaN,Low
1411,FKO001162,Delhi,2026-01-22,2026-01-28,NaT,NaN,Low
1452,FKO001245,Ahmedabad,2026-06-08,2026-06-14,NaT,NaN,Low



Original Cleaned Data Check:


,order_id,order_status,promised_delivery_date,actual_delivery_date
27,FKO001245,Delivered,2026-06-14,NaT
44,FKO000664,Delivered,2026-01-20,NaT
320,FKO000913,Delivered,2026-02-05,NaT
380,FKO000830,Delivered,2026-05-01,NaT
533,FKO000166,Delivered,2026-06-17,NaT
919,FKO000581,Delivered,2026-01-31,NaT
1052,FKO001162,Delivered,2026-01-28,NaT
1058,FKO000249,Delivered,2026-01-27,NaT
1214,FKO001411,Delivered,2026-02-27,NaT
1316,FKO000332,Delivered,2026-02-17,NaT


Missing actual delivery dates in orders_clean: 13

Dataset Shapes:
Orders: (1600, 9)
Refunds: (1350, 5)
Complaints: (1500, 6)
Final Dataset: (1600, 21)
Final Report: (1600, 19)

Export File Checks:
flipkart_order_issue_report.csv Exists: True
flipkart_issue_summary.xlsx Exists: True

Final Priority Counts:
support_priority
Low       688
Medium    678
High      234
Name: count, dtype: int64

Final report validation checks completed.


In [45]:
from pathlib import Path

output_files = [
    "flipkart_order_issue_report.csv",
    "flipkart_issue_summary.xlsx"
]

for filename in output_files:
    file_path = Path("/content") / filename

    if file_path.exists():
        print(f"Generated: {filename}")
        print(f"File size: {file_path.stat().st_size:,} bytes")
    else:
        print(f"Not found: {filename}")

Generated: flipkart_order_issue_report.csv
File size: 194,296 bytes
Generated: flipkart_issue_summary.xlsx
File size: 50,548 bytes


In [47]:
from google.colab import files

files.download("/content/flipkart_order_issue_report.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [49]:
from google.colab import files

files.download("/content/flipkart_issue_summary.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Submission handover
1. Save your completed Colab copy in Drive and share its link with the required permissions, or submit the `.ipynb` file if instructed by your instructor.
2. Include the final automated scorecard **and** your written findings in the notebook.
3. Do not submit the original three input datasets unless requested; output CSV and Excel files are generated during your run.
